In [4]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# veri Setlerini Yükleme
print("Veriler yükleniyor...")
df_raw = pd.read_csv('results.csv')
df_team_stats = pd.read_csv('wc_team_alltime_stats.csv')

df_raw['date'] = pd.to_datetime(df_raw['date'])

# resmi ve rekabetçi turnuvaları filtreleme
official_tournaments = [
    'FIFA World Cup', 'UEFA Euro', 'Copa América', 'AFC Asian Cup', 
    'African Cup of Nations', 'FIFA World Cup qualification', \
    'UEFA Euro qualification', 'UEFA Nations League'
]
df_master = df_raw[df_raw['tournament'].isin(official_tournaments)].copy()
df_master = df_master.sort_values(by='date').reset_index(drop=True)

# beraberlikleri eleme
df_master = df_master[df_master['home_score'] != df_master['away_score']].copy()
df_master['result'] = np.where(df_master['home_score'] > df_master['away_score'], 1, 2) # 1: Ev, 2: Deplasman

print(f"Beraberlikler elendikten sonra kalan net maç sayısı: {len(df_master)}")

# son 5 maçlık form ve gol trendleri hesaplama
print("Son 5 maçlık form ve gol trendleri hesaplanıyor (Bu işlem biraz sürebilir)...")

home_form_list, away_form_list = [], []
home_avg_scored_list, away_avg_scored_list = [], []
home_avg_conceded_list, away_avg_conceded_list = [], []

for idx, row in df_master.iterrows():
    match_date = row['date']
    h_team = row['home_team']
    a_team = row['away_team']
    
    # o tarihten önceki maç havuzu
    past_matches = df_master[df_master['date'] < match_date]
    
    h_past = past_matches[(past_matches['home_team'] == h_team) | (past_matches['away_team'] == h_team)].tail(5)
    a_past = past_matches[(past_matches['home_team'] == a_team) | (past_matches['away_team'] == a_team)].tail(5)
    
    # ev sahibi form hesaplama
    if len(h_past) > 0:
        h_scored = np.where(h_past['home_team'] == h_team, h_past['home_score'], h_past['away_score'])
        h_conceded = np.where(h_past['home_team'] == h_team, h_past['away_score'], h_past['home_score'])
        h_wins = np.where(h_past['home_team'] == h_team, h_past['result'] == 1, h_past['result'] == 2)
        
        home_form_list.append(np.sum(h_wins) / len(h_past))
        home_avg_scored_list.append(np.mean(h_scored))
        home_avg_conceded_list.append(np.mean(h_conceded))
    else:
        home_form_list.append(0.5)
        home_avg_scored_list.append(1.0)
        home_avg_conceded_list.append(1.0)
        
    # deplasman form hesaplama
    if len(a_past) > 0:
        a_scored = np.where(a_past['home_team'] == a_team, a_past['home_score'], a_past['away_score'])
        a_conceded = np.where(a_past['home_team'] == a_team, a_past['away_score'], a_past['home_score'])
        a_wins = np.where(a_past['home_team'] == a_team, a_past['result'] == 1, a_past['result'] == 2)
        
        away_form_list.append(np.sum(a_wins) / len(a_past))
        away_avg_scored_list.append(np.mean(a_scored))
        away_avg_conceded_list.append(np.mean(a_conceded))
    else:
        away_form_list.append(0.5)
        away_avg_scored_list.append(1.0)
        away_avg_conceded_list.append(1.0)

df_master['home_form'] = home_form_list
df_master['away_form'] = away_form_list
df_master['home_avg_scored'] = home_avg_scored_list
df_master['away_avg_scored'] = away_avg_scored_list
df_master['home_avg_conceded'] = home_avg_conceded_list
df_master['away_avg_conceded'] = away_avg_conceded_list

# tarihsel turnuva başarılarını bağlama
df_team_stats_filtered = df_team_stats[['team', 'total_wc_appearances', 'titles', 'finals_reached', 'semis_reached']]

df_master = pd.merge(df_master, df_team_stats_filtered, left_on='home_team', right_on='team', how='left').rename(columns={
    'total_wc_appearances': 'home_wc_appearances', 'titles': 'home_titles', 'finals_reached': 'home_finals_reached', 'semis_reached': 'home_semis_reached'
}).drop(columns=['team']).fillna(0)

df_master = pd.merge(df_master, df_team_stats_filtered, left_on='away_team', right_on='team', how='left').rename(columns={
    'total_wc_appearances': 'away_wc_appearances', 'titles': 'away_titles', 'finals_reached': 'away_finals_reached', 'semis_reached': 'away_semis_reached'
}).drop(columns=['team']).fillna(0)

# özellik havuzu ve hedef değişken seçimi
final_features = [
    'home_form', 'away_form', 'home_avg_scored', 'away_avg_scored', 'home_avg_conceded', 'away_avg_conceded',
    'home_wc_appearances', 'away_wc_appearances', 'home_titles', 'away_titles', 'home_finals_reached', 'away_finals_reached'
]

X = df_master[final_features]
y = df_master['result']

# %80 train %20 test olarak split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model eğitimi catboost ile
print("Nihai Model Eğitiliyor...")
production_model = CatBoostClassifier(iterations=500, learning_rate=0.03, depth=6, random_seed=42, verbose=0)
production_model.fit(X_train, y_train)

# skor ölçümü 
y_pred = production_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n GÜÇLENDİRİLMİŞ ÜRETİM MODELİ ACCURACY SKORU: %{accuracy * 100:.2f}")
print("\n--- Detaylı Model Raporu ---")
print(classification_report(y_test, y_pred))

# modeli ve güncel verileri diske kaydetme
joblib.dump(production_model, 'football_production_model.pkl')
df_master.to_csv('final_football_data.csv', index=False)
print("\n[BAŞARILI] 'football_production_model.pkl' ve 'final_football_data.csv' başarıyla oluşturuldu!")

Veriler yükleniyor...
Beraberlikler elendikten sonra kalan net maç sayısı: 12379
Son 5 maçlık form ve gol trendleri hesaplanıyor (Bu işlem biraz sürebilir)...
Nihai Model Eğitiliyor...

 GÜÇLENDİRİLMİŞ ÜRETİM MODELİ ACCURACY SKORU: %77.30

--- Detaylı Model Raporu ---
              precision    recall  f1-score   support

           1       0.80      0.86      0.83      1588
           2       0.71      0.62      0.66       888

    accuracy                           0.77      2476
   macro avg       0.76      0.74      0.75      2476
weighted avg       0.77      0.77      0.77      2476


[BAŞARILI] 'football_production_model.pkl' ve 'final_football_data.csv' başarıyla oluşturuldu!
